In [ ]:
import os
os.environ['AWS_PROFILE'] = 'admin'
os.environ['HAVEN_DATABASE'] = 'haven'

import plotly.graph_objects as go
import plotly.express as px
import pandas as pd
import numpy as np
import h3
from tqdm import tqdm

from mirrorverse.utils import read_data_w_cache
from mirrorverse.plotting import build_geojson
from haven.db import write_data

from sklearn.decomposition import PCA
from multiprocessing import Pool

In [ ]:
sql = '''
select distinct
    extract(year from time) as year 
from 
    copernicus_biochemistry
'''
years = sorted(read_data_w_cache(sql)['year'])
print(years)

In [ ]:
for year in tqdm(years):
    sql = f'''
    select
        h3_index,
        chlorophyll, 
        time,
        extract(year from time) as year,
        extract(month from time) as month
    from 
        copernicus_biochemistry
    where 
        h3_resolution = 4
        and depth_bin = 25.0
        and extract(year from time) = {year}
        and extract(day from time) = 1
    '''
    raw_data = read_data_w_cache(sql)
    raw_data['lat'] = raw_data['h3_index'].apply(lambda h: h3.h3_to_geo(h)[0])
    raw_data['lon'] = raw_data['h3_index'].apply(lambda h: h3.h3_to_geo(h)[1])
    raw_data['epoch'] = raw_data['time'].astype('int64') // 10**9
    raw_data['raw_chlorophyll'] = raw_data['chlorophyll']
    raw_data['chlorophyll'] = (raw_data['raw_chlorophyll'] + 1) ** 0.25
    raw_data = raw_data.sort_values(['epoch', 'h3_index'], ascending=False).reset_index(drop=True)
    raw_data.to_csv(f'raw_chlorophyll_{year}.csv.gz', index=False)

In [ ]:
dfs = []
for year in tqdm(years):
    dfs.append(pd.read_csv(f'raw_chlorophyll_{year}.csv.gz'))
data = pd.concat(dfs).sort_values(['h3_index', 'epoch'], ascending=True).reset_index(drop=True)[['h3_index', 'chlorophyll', 'year', 'month']]
print(data.shape)
data.head()

In [ ]:
def split(raw_data, num_splits):
    h3_indices = set(raw_data['h3_index'])
    splits = [set() for _ in range(num_splits)]
    for i, h3_index in enumerate(h3_indices):
        splits[i % num_splits].add(h3_index)
    return [
        raw_data[raw_data['h3_index'].isin(splits[i])]
        for i in range(num_splits)
    ]

def job(data):
    rows = []
    for h3_index in data['h3_index'].unique():
        df = data[data['h3_index'] == h3_index]
        row = {
            str(i): value 
            for i, value in enumerate(df['chlorophyll'].values)
        }
        row['h3_index'] = h3_index
        rows.append(row)
    return pd.DataFrame(rows)
        

num_splits = 8
with Pool(num_splits) as p:
    dfs = p.map(job, split(data, num_splits))
timelines = pd.concat(dfs).set_index('h3_index')

In [ ]:
X = np.array(timelines)
X_mean = np.mean(X, axis=0)
X = X - X_mean
pca = PCA(n_components=3)
pca.fit(X)
print(pca.explained_variance_ratio_.sum())
pca.explained_variance_ratio_

In [ ]:
loadings = timelines.copy() 
for i in range(3):
    loadings[f'component_{i}'] = pca.transform(X)[:,i]
    loadings = loadings[[c for c in loadings.columns if c.startswith('component')]]
loadings = loadings.reset_index()
loadings.head()

In [ ]:
df = loadings.copy()
df['lat'] = df['h3_index'].apply(lambda x: h3.h3_to_geo(x)[0])
df['lon'] = df['h3_index'].apply(lambda x: h3.h3_to_geo(x)[1])
df = df[(df['lon'] > -170) & (df['lon'] < 0) & (df['lat'] > 42) & (df['lat'] < 64)]

fig = go.Figure()
geojson = build_geojson(df, 'h3_index')
fig.add_trace(
    go.Choroplethmapbox(
        geojson=geojson,
        locations=df['h3_index'],
        z=df['component_2'],
        visible=True,
        marker_line_color='rgba(255,255,255,0)',
        colorscale='algae'
    )
)
fig.update_layout(
    autosize=False,  # Disable autosizing
    width=800,       # Set width in pixels
    height=800,      # Set height in pixels
)
fig.update_layout(
    margin={"r":0,"t":30,"l":0,"b":0}, mapbox=dict(style="carto-positron", zoom=3, center = {"lat": 57, "lon": -150})
)
fig.show()

In [ ]:
loadings['version'] = 1
write_data(loadings, 'static_pca_habitat', ['version'])